# 📋 Notebook 1 — Summary Quality Verification

Verify all 85 docs were summarised, check word counts, read any summary.

**Works from any directory** — path detection is automatic.

In [1]:
import os
from pathlib import Path
import pandas as pd

def find_backend() -> Path:
    """Walk up from cwd until we find the backend root."""
    for p in [Path(os.getcwd()).resolve()] + list(Path(os.getcwd()).resolve().parents):
        if (p / 'parent_docstore').exists() and (p / 'data').exists():
            return p
        # Check inside a 'backend' subfolder
        if (p / 'backend' / 'parent_docstore').exists():
            return p / 'backend'
    raise RuntimeError('Cannot find backend root. Make sure ingest.py has been run.')

BACKEND           = find_backend()
SUMMARIES_PATH    = BACKEND / 'summaries'
PARENT_STORE_PATH = BACKEND / 'parent_docstore'

print(f'✅ Backend root : {BACKEND}')
print(f'   Summaries   : {SUMMARIES_PATH}  ({len(list(SUMMARIES_PATH.glob("*.txt")))} files)')
print(f'   Parent store: {PARENT_STORE_PATH}  (exists={PARENT_STORE_PATH.exists()})')

✅ Backend root : C:\Users\karth\nexora-sentiobot\backend
   Summaries   : C:\Users\karth\nexora-sentiobot\backend\summaries  (85 files)
   Parent store: C:\Users\karth\nexora-sentiobot\backend\parent_docstore  (exists=True)


In [2]:
from langchain.storage import LocalFileStore
from langchain.storage._lc_store import create_kv_docstore

byte_store = LocalFileStore(str(PARENT_STORE_PATH))
docstore   = create_kv_docstore(byte_store)
all_ids    = list(byte_store.yield_keys())

rows = []
for f in sorted(SUMMARIES_PATH.glob('*.txt')):
    doc_id = f.stem
    text   = f.read_text(encoding='utf-8').strip()
    parent = docstore.mget([doc_id])[0]
    if parent:
        source  = parent.metadata.get('source', '?')
        section = (parent.metadata.get('section_title')
                   or parent.metadata.get('Category')
                   or '?')
    else:
        source, section = '?', '?'
    rows.append({
        'doc_id':     doc_id[:16] + '…',
        'source':     source,
        'section':    str(section)[:50],
        'word_count': len(text.split()),
        'summary':    text,
    })

df = pd.DataFrame(rows)
print(f'Total summaries : {len(df)}')
print(f'Total parents   : {len(all_ids)}')
print(f'Missing         : {len(all_ids) - len(df)}')
df[['source','section','word_count']].head(10)

Total summaries : 85
Total parents   : 85
Missing         : 0


,source,section,word_count
0,policies.md,5. End-of-Life (EOL) Policy,48
1,faqs.csv,Lighting,37
2,faqs.csv,App,34
3,faqs.csv,Thermostat,32
4,faqs.csv,Camera,42
5,faqs.csv,Lighting,34
6,nexora_thermostat_pro_manual.md,7. In-Depth Troubleshooting,105
7,visionsphere_360_manual.md,4. Installation & Initial Setup,86
8,faqs.csv,Returns,43
9,faqs.csv,Lighting,45


In [3]:
# Word count distribution
print(df['word_count'].describe().round(1).to_string())
thin = df[df['word_count'] < 30]
print()
if thin.empty:
    print('✅  All summaries >= 30 words')
else:
    print(f'⚠️  {len(thin)} suspiciously short:')
    print(thin[['source','section','word_count']].to_string())

count     85.0
mean      57.4
std       27.1
min       31.0
25%       37.0
50%       47.0
75%       68.0
max      138.0

✅  All summaries >= 30 words


In [4]:
# Coverage per source
print(df.groupby('source').agg(
    summaries=('doc_id','count'),
    avg_words=('word_count','mean'),
    min_words=('word_count','min'),
    max_words=('word_count','max'),
).round(1).to_string())

                                   summaries  avg_words  min_words  max_words
source                                                                       
faqs.csv                                  50       54.1         31        133
lumiglow_smart_lighting_manual.md          9       56.0         32        120
nexora_thermostat_pro_manual.md           10       70.8         42        138
policies.md                                7       63.7         39         92
visionsphere_360_manual.md                 9       57.9         34         86


In [5]:
# Read any summary in full — change these filters
FILTER_SOURCE  = 'policies.md'
FILTER_SECTION = 'Warranty'

matches = df[
    df['source'].str.contains(FILTER_SOURCE, case=False) &
    df['section'].str.contains(FILTER_SECTION, case=False)
]
if matches.empty:
    print('No match. Available sections:')
    print(df[df['source'].str.contains(FILTER_SOURCE, case=False)]['section'].tolist())
else:
    for _, row in matches.iterrows():
        print(f"Source: {row['source']}  |  Section: {row['section']}  |  Words: {row['word_count']}")
        print()
        print(row['summary'])
        print('─'*60)

Source: policies.md  |  Section: 1. Limited Warranty Policy  |  Words: 63

The Limited Warranty Policy for Nexora products, including the Nexora Thermostat Pro with a 2-Year Limited Warranty, SecureSphere 360 Camera with a 1-Year Limited Warranty, and LumiGlow Smart Lights with a 1-Year Limited Warranty, covers hardware failures and firmware malfunctions under normal use conditions, but excludes damage caused by accident, abuse, misuse, or external causes, and requires proof of purchase for warranty claims
────────────────────────────────────────────────────────────


In [6]:
# Find any missing docs
existing = {f.stem for f in SUMMARIES_PATH.glob('*.txt')}
missing  = [i for i in all_ids if i not in existing]
if not missing:
    print('✅  100% coverage')
else:
    print(f'❌  {len(missing)} missing:')
    for doc in docstore.mget(missing):
        if doc:
            print(f"  • {doc.metadata.get('source')} › {doc.metadata.get('section_title','?')}")

✅  100% coverage
